# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv


In [ ]:
# The command %run update_path.py runs a local script that adds 
# the repository's ../../05_src/ directory to the notebook's kernel path - 
# so that we can use our modules within the notebook.
%run ../../01_materials/labs/update_path.py

This code sets up the environment for a Python script by importing required libraries and initializing a module-level logger.
* Imports utility tools: It loads glob (for finding file pathnames), os (for interacting with the operating system), pandas (for data manipulation), and random (for generating random numbers).
* Initializes a logger: It imports a custom logging function (get_logger) from utils.logger and uses it to create a logger instance (_logs).


In [20]:
from glob import glob
import os
import pandas as pd
import random

from utils.logger import get_logger
_logs = get_logger(__name__)

We sample a subset of the files in the directory below.

In [21]:
stock_files = glob(os.path.join(os.getenv('SRC_DIR'), "data/prices_csv/stocks/*.csv"))
_logs.info(f'Found {len(stock_files)} stock price files.')

random.seed(42)
n_sample = 60
stock_files = random.sample(stock_files, n_sample)
_logs.info(f'Sampled {n_sample} stock price files for processing. The files are: {stock_files   }')

2026-10-09 14:37:30,752, 3619788285.py, 2, INFO, Found 5884 stock price files.
2026-10-09 14:37:30,754, 3619788285.py, 7, INFO, Sampled 60 stock price files for processing. The files are: ['../../05_src/data/prices_csv/stocks\\TNC.csv', '../../05_src/data/prices_csv/stocks\\CBB.csv', '../../05_src/data/prices_csv/stocks\\ALDX.csv', '../../05_src/data/prices_csv/stocks\\GLADD.csv', '../../05_src/data/prices_csv/stocks\\FIXX.csv', '../../05_src/data/prices_csv/stocks\\ETJ.csv', '../../05_src/data/prices_csv/stocks\\CMCTP.csv', '../../05_src/data/prices_csv/stocks\\BWG.csv', '../../05_src/data/prices_csv/stocks\\VIAC.csv', '../../05_src/data/prices_csv/stocks\\REI.csv', '../../05_src/data/prices_csv/stocks\\BLPH.csv', '../../05_src/data/prices_csv/stocks\\SMG.csv', '../../05_src/data/prices_csv/stocks\\MOH.csv', '../../05_src/data/prices_csv/stocks\\AMH.csv', '../../05_src/data/prices_csv/stocks\\AMAL.csv', '../../05_src/data/prices_csv/stocks\\BPYPN.csv', '../../05_src/data/prices_csv/st

This code loops through a list of stock CSV files, reads them into pandas DataFrames, adds metadata columns, and combines them into a single dataset.

In [22]:
dt_list = []
for s_file in stock_files:
    _logs.info(f"Reading file: {s_file}")
    dt = pd.read_csv(s_file).assign(
        source = os.path.basename(s_file),
        ticker = os.path.basename(s_file).replace('.csv', ''),
        Date = lambda x: pd.to_datetime(x['Date'])
    )
    dt_list.append(dt)
stock_prices = pd.concat(dt_list, axis = 0, ignore_index = True)

2026-10-09 14:38:03,709, 2577063914.py, 3, INFO, Reading file: ../../05_src/data/prices_csv/stocks\TNC.csv
2026-10-09 14:38:03,734, 2577063914.py, 3, INFO, Reading file: ../../05_src/data/prices_csv/stocks\CBB.csv
2026-10-09 14:38:03,747, 2577063914.py, 3, INFO, Reading file: ../../05_src/data/prices_csv/stocks\ALDX.csv
2026-10-09 14:38:03,757, 2577063914.py, 3, INFO, Reading file: ../../05_src/data/prices_csv/stocks\GLADD.csv
2026-10-09 14:38:03,765, 2577063914.py, 3, INFO, Reading file: ../../05_src/data/prices_csv/stocks\FIXX.csv
2026-10-09 14:38:03,769, 2577063914.py, 3, INFO, Reading file: ../../05_src/data/prices_csv/stocks\ETJ.csv
2026-10-09 14:38:03,780, 2577063914.py, 3, INFO, Reading file: ../../05_src/data/prices_csv/stocks\CMCTP.csv
2026-10-09 14:38:03,791, 2577063914.py, 3, INFO, Reading file: ../../05_src/data/prices_csv/stocks\BWG.csv
2026-10-09 14:38:03,806, 2577063914.py, 3, INFO, Reading file: ../../05_src/data/prices_csv/stocks\VIAC.csv
2026-10-09 14:38:03,816, 25770

This code clears and recreates a specific folder to prepare a clean file path for storing a CSV file

In [24]:
import shutil

temp = os.getenv("TEMP_DATA")
csv_dir = os.path.join(temp, "csv")

shutil.rmtree(csv_dir, ignore_errors=True)
os.makedirs(csv_dir, exist_ok=True)
stock_csv = os.path.join(csv_dir, "stock_px.csv")

This code prepares a clean directory for storing Parquet files.

In [25]:
import dask.dataframe as dd

parquet_dir = os.path.join(temp, "parquet")
shutil.rmtree(parquet_dir, ignore_errors=True)
os.makedirs(parquet_dir, exist_ok=True)

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [26]:
# Clean up before start
PRICE_DATA = os.getenv("PRICE_DATA")
import shutil
if os.path.exists(PRICE_DATA):
    shutil.rmtree(PRICE_DATA)

This code splits a stock price dataset into smaller files organized by ticker and year.

In [27]:
for ticker in stock_prices['ticker'].unique():
    # Filter data for ticker
    # Notice that these are Pandas dataframes
    _logs.info(f'Processing ticker: {ticker}')
    ticker_dt = stock_prices[stock_prices['ticker'] == ticker]
    ticker_dt = ticker_dt.assign(Year = ticker_dt.Date.dt.year)
    for yr in ticker_dt['Year'].unique():
        _logs.info(f'Processing year {yr} for ticker {ticker}.')
        # Filter data for year and convert to Dask dataframe
        yr_dd = dd.from_pandas(ticker_dt[ticker_dt['Year'] == yr],2)
        
        # Define path and create directories if not exist
        yr_path = os.path.join(PRICE_DATA, ticker, f"{ticker}_{yr}")
        os.makedirs(os.path.dirname(yr_path), exist_ok=True)
        _logs.info(f'Writing data to path: {yr_path}')

        # Write to Parquet
        yr_dd.to_parquet(yr_path, engine = "pyarrow")

2026-10-09 14:41:33,803, 2418108412.py, 4, INFO, Processing ticker: TNC
2026-10-09 14:41:33,817, 2418108412.py, 8, INFO, Processing year 1973 for ticker TNC.
2026-10-09 14:41:33,826, 2418108412.py, 15, INFO, Writing data to path: ../../05_src/data/prices/TNC\TNC_1973
2026-10-09 14:41:33,880, 2418108412.py, 8, INFO, Processing year 1974 for ticker TNC.
2026-10-09 14:41:33,885, 2418108412.py, 15, INFO, Writing data to path: ../../05_src/data/prices/TNC\TNC_1974
2026-10-09 14:41:33,898, 2418108412.py, 8, INFO, Processing year 1975 for ticker TNC.
2026-10-09 14:41:33,900, 2418108412.py, 15, INFO, Writing data to path: ../../05_src/data/prices/TNC\TNC_1975
2026-10-09 14:41:33,911, 2418108412.py, 8, INFO, Processing year 1976 for ticker TNC.
2026-10-09 14:41:33,915, 2418108412.py, 15, INFO, Writing data to path: ../../05_src/data/prices/TNC\TNC_1976
2026-10-09 14:41:33,926, 2418108412.py, 8, INFO, Processing year 1977 for ticker TNC.
2026-10-09 14:41:33,929, 2418108412.py, 15, INFO, Writing 

In [28]:
import os
from glob import glob

# Write your code below.
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
_logs.info(f'Found {len(parquet_files)} parquet files for reading back into Dask.')

dd_px = dd.read_parquet(parquet_files).set_index("ticker")


2026-10-09 14:42:31,218, 2043994053.py, 6, INFO, Found 2045 parquet files for reading back into Dask.


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.
dd_shift = (
    dd_px
        .groupby('ticker', group_keys=False)
        .apply(
            lambda x: x.sort_values('Date', ascending=True)
                       .assign(
                           Close_lag_1 = x['Close'].shift(1),
                           Adj_Close_lag_1 = x['Adj Close'].shift(1)
                       ),
            meta = pd.DataFrame(data ={'Date': 'datetime64[ns]',
                    'Open': 'f8',
                    'High': 'f8',
                    'Low': 'f8',
                    'Close': 'f8',
                    'Adj Close': 'f8',
                    'Volume': 'i8',
                    'source': 'object',
                    'Year': 'int32',
                    'Close_lag_1': 'f8',
                    'Adj_Close_lag_1': 'f8'},
                    index = pd.Index([], dtype=pd.StringDtype(), name='ticker'))
    ))


In [ ]:
dd_feat = dd_shift.assign(
    returns = lambda x: x['Close'] / x['Close_lag_1'] - 1,
    hi_lo_range = lambda x: x['High'] - x['Low']
)

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# Write your code below.
feat_pdf = dd_feat.compute()

feat_pdf = feat_pdf.sort_values('Date')
feat_pdf['returns_ma_10'] = (
    feat_pdf.groupby('ticker')['returns']
            .transform(lambda x: x.rolling(10).mean())
)

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

**Was it necessary to convert to pandas to calculate the moving average return?**

It was not necessary, but it was the simplest option here. Dask is better on storage but too complex for this scenario. Since our sample fits in memory, converting to pandas is the better choice.

**Would it have been better to do it in Dask? Why?**

My opinion is that pandas was the better choice given the data size. My understaning of the lectures is that Dask's main benefit is handling data that doesn't fit in memory and parallelizing across partitions is not worth it for a dataset this small. Dask would be the better choice only if the dataset were too large to fit in memory and then we can make use of Dask's partitioning capabilities instead of pulling everything into a single pandas dataframe.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.